# TP — Prediction de la direction des marches (Trading) — **LSTM / Keras**

**M2 DMIA — Deep Learning — Session 4 (Projet 3 : la memoire des sequences)**

**Question** : a partir des **N derniers jours** de cours d'un actif (action, indice ou crypto),
**le prix montera-t-il ou baissera-t-il demain** ? (hausse / baisse → classification binaire).

C'est l'exemple canonique de la **memoire** : pour decider, le reseau doit « se souvenir » de la
tendance recente. C'est exactement ce que fait un **LSTM** (Long Short-Term Memory).

On suit le **pipeline DL** standard en 7 etapes :
1. Charger les donnees → 2. Construire les features (sequences) → 3. Pretraiter (split chronologique) →
4. Definir le modele → 5. Entrainer → 6. Evaluer → 7. Analyser et discuter.

---

> ## AVERTISSEMENT pedagogique (a lire absolument)
>
> Ce notebook est un **exercice d'apprentissage du Deep Learning**, **PAS un conseil d'investissement**.
>
> - Les marches financiers sont **extremement bruites** : la majorite du mouvement quotidien est
>   imprevisible (proche d'une marche aleatoire).
> - Un modele qui atteint **~55 % d'accuracy** sur ce probleme est deja **tres bon** — on est loin
>   des 99 % de MNIST. Ne soyez pas decu par un score « mediocre » : c'est la nature du probleme.
> - Nous insistons sur deux **pieges methodologiques** mortels :
>   - le **data leakage** (utiliser le futur pour preparer le passe),
>   - le **sur-apprentissage** (le modele memorise le bruit du passe au lieu d'apprendre un signal).
> - Un bon score **sur des donnees passees** ne garantit **rien** sur le futur : c'est l'**illusion
>   de la retro-prediction**.
>
> L'objectif est de **comprendre le LSTM** ET de **developper un esprit critique** sur les metriques.

## 1. Imports et reproductibilite

On charge les bibliotheques standard. La **graine (seed)** garantit que les memes resultats
sont reproductibles d'une execution a l'autre.

> Recommande : executer sur **Google Colab** (Execution → Modifier le type d'execution → GPU).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, accuracy_score

# Reproductibilite (la meme graine -> les memes resultats)
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

print('TensorFlow version :', tf.__version__)

## 2. Charger les prix (donnees REELLES + repli SYNTHETIQUE automatique)

**Principe de robustesse** : on tente d'abord de telecharger des **donnees reelles** avec
`yfinance` (gratuit, sans cle d'API). Si la bibliotheque est absente ou si le telechargement
echoue (pas de reseau, API capricieuse...), on bascule **automatiquement** sur un **repli
synthetique**.

### Pourquoi un repli synthetique « avec signal » ?

Si on generait un prix purement aleatoire (marche aleatoire pure), **aucun** modele ne pourrait
rien apprendre : la direction de demain serait totalement independante du passe. Pour que le TP
reste pedagogique **meme hors-ligne**, le repli ajoute une **legere tendance** et une **composante
saisonniere** (un cycle). Resultat : un signal **partiellement** predictible — exactement le genre
de structure faible qu'un LSTM peut tenter de capter.

In [ ]:
# Cellule optionnelle : installer yfinance si absent (utile sur Colab).
# Entoure de try/except : si l'installation echoue, le repli synthetique prend le relais.
try:
    import yfinance  # noqa: F401
except Exception:
    try:
        import subprocess, sys
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'yfinance'], check=False)
    except Exception as e:
        print('Installation de yfinance impossible (on utilisera le repli) :', repr(e))

In [ ]:
# --- Tentative 1 : donnees REELLES via yfinance ---
# (entoure de try/except pour rester executable hors-ligne)

TICKER = '^GSPC'   # S&P 500. Essayez aussi 'BTC-USD' (crypto) ou 'AAPL' (action).
START = '2015-01-01'

prices = None
source = None

try:
    import yfinance as yf
    df_yf = yf.download(TICKER, start=START, progress=False)
    if df_yf is not None and len(df_yf) > 300:
        close = df_yf['Close']                       # prix de cloture journalier
        prices = np.asarray(close).reshape(-1).astype('float64')
        source = 'REELLES (yfinance: ' + TICKER + ')'
except Exception as e:
    print('yfinance indisponible ou echec du telechargement :', repr(e))

# --- Tentative 2 : repli SYNTHETIQUE ---
if prices is None:
    n_days = 2500
    rng = np.random.default_rng(SEED)

    # 1) Marche aleatoire geometrique avec une LEGERE tendance haussiere
    mu = 0.0003          # derive (tendance) journaliere moyenne
    sigma = 0.01         # volatilite journaliere (bruit)
    daily_noise = rng.normal(mu, sigma, size=n_days)

    # 2) Composante SAISONNIERE : un cycle lent ajoute du signal predictible
    t = np.arange(n_days)
    seasonal = 0.0015 * np.sin(2 * np.pi * t / 60.0)   # cycle ~60 jours

    daily_returns_syn = daily_noise + seasonal
    prices = 100.0 * np.exp(np.cumsum(daily_returns_syn))   # prix = produit cumule des rendements
    source = 'SYNTHETIQUE (repli automatique)'

# Message clair sur l'origine des donnees
if source.startswith('REELLES'):
    print('Donnees REELLES (yfinance: ' + TICKER + ')')
else:
    print('Repli SYNTHETIQUE')
print('Nombre de jours :', len(prices))
print('Prix (min/max)  : %.2f / %.2f' % (prices.min(), prices.max()))

In [ ]:
# Tracons la courbe de prix
plt.figure(figsize=(11, 4))
plt.plot(prices, color='steelblue', linewidth=1)
plt.title('Cours de cloture journalier  [source : ' + source + ']')
plt.xlabel('jour')
plt.ylabel('prix')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Features : rendements et sequences glissantes

### Pourquoi travailler sur les RENDEMENTS et non les prix bruts ?

Le prix brut est **non-stationnaire** : sa moyenne et sa variance derivent dans le temps (un prix
qui vaut 100 puis 4000 n'a pas la meme echelle). Les reseaux apprennent mal sur des series dont
les statistiques changent.

Le **rendement journalier** `r_t = (P_t - P_{t-1}) / P_{t-1}` (la variation **relative**) est
beaucoup plus **stationnaire** : il oscille autour de 0 avec une echelle stable. C'est la
grandeur naturelle pour ce probleme.

### Construction des sequences

- On choisit une fenetre de **N = 20 jours** (un mois de bourse environ).
- Chaque **echantillon** = les rendements des N derniers jours : `[r_{t-N+1}, ..., r_t]`.
- La **cible** = **1** si le rendement du **jour suivant** est positif (hausse), **0** sinon (baisse).

C'est la **fenetre glissante** (sliding window) : on fait glisser la fenetre d'un jour a la fois.

In [ ]:
# 1) Rendements journaliers (variation relative) via pandas pct_change
returns = pd.Series(prices).pct_change().dropna().to_numpy()
print('Nombre de rendements :', len(returns))
print('Rendement moyen      : %.5f' % returns.mean())
print('Ecart-type           : %.5f' % returns.std())

# 2) Fenetre glissante : sequences de N jours -> cible binaire (hausse de demain)
N = 20   # nombre de jours d'historique par sequence

X, y = [], []
for i in range(N, len(returns)):
    X.append(returns[i - N:i])            # les N rendements precedents
    y.append(1 if returns[i] > 0 else 0)  # 1 = hausse le jour i, 0 = baisse

X = np.array(X, dtype='float32')
y = np.array(y, dtype='int32')

# Keras LSTM attend la forme (echantillons, pas_de_temps, features)
X = X.reshape((X.shape[0], N, 1))

print('X :', X.shape, '| y :', y.shape)
print('Proportion de hausses (classe 1) : %.3f' % y.mean())

**Note** : regardez la proportion de hausses ci-dessus. Sur la plupart des actifs, elle est
proche de **0.5** (un peu plus a cause de la tendance haussiere de long terme). Cela montre deja
qu'une **baseline naive** « toujours predire hausse » obtient un score proche de cette proportion —
c'est le score qu'il faudra **battre**.

## 4. Pretraiter : split CHRONOLOGIQUE et normalisation (sans fuite du futur !)

### Regle d'or n.1 : JAMAIS de `shuffle` sur une serie temporelle

Pour une serie temporelle, l'ordre est **sacre**. On entraine sur le **passe** et on teste sur le
**futur**. Melanger (shuffle) les jours reviendrait a entrainer le modele avec des informations du
futur — un **data leakage** qui gonfle artificiellement le score. On fait donc un **split
chronologique** : les premiers 80 % en entrainement, les derniers 20 % en test.

### Regle d'or n.2 : la normalisation se calcule sur le TRAIN SEULEMENT

On standardise les rendements (moyenne 0, ecart-type 1). **Crucial** : la moyenne et l'ecart-type
sont calcules **uniquement sur le train**, puis appliques au test. Utiliser les statistiques de
**tout** le dataset (train + test) ferait fuiter de l'information du futur dans la normalisation —
encore du **data leakage**.

In [ ]:
# Split CHRONOLOGIQUE (surtout pas de shuffle !)
split = int(0.80 * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

print('Train :', X_train.shape, '| Test :', X_test.shape)

# Normalisation : statistiques calculees sur le TRAIN UNIQUEMENT
mu_train = X_train.mean()
sigma_train = X_train.std() + 1e-8   # +epsilon pour eviter la division par 0

X_train = (X_train - mu_train) / sigma_train
X_test = (X_test - mu_train) / sigma_train   # on applique les stats du TRAIN au test

print('Apres normalisation (train) : moyenne = %.4f, ecart-type = %.4f' % (X_train.mean(), X_train.std()))
print('Stats utilisees -> mu_train = %.6f, sigma_train = %.6f' % (mu_train, sigma_train))

## 5. Definir le modele : un LSTM

- **LSTM(64)** : la couche a memoire. Elle parcourt la sequence de 20 jours et maintient un
  « etat » resumant ce qu'elle a vu (la tendance recente). C'est la **memoire** des sequences.
- **Dropout(0.3)** : on « eteint » aleatoirement 30 % des neurones a l'entrainement pour
  **limiter le sur-apprentissage** (essentiel ici, car le bruit est facile a memoriser).
- **Dense(1, sigmoid)** : un seul neurone de sortie donnant la **probabilite de hausse** (entre 0 et 1).

Perte **binary_crossentropy** (classification binaire), optimiseur **Adam**, metrique **accuracy**.

In [ ]:
model = keras.Sequential([
    layers.Input(shape=(N, 1)),             # sequence de N pas de temps, 1 feature (le rendement)
    layers.LSTM(64),                        # couche a memoire
    layers.Dropout(0.3),                    # regularisation anti sur-apprentissage
    layers.Dense(1, activation='sigmoid')   # probabilite de hausse
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

## 6. Entrainer

On entraine sur quelques epoques, avec une part de **validation** prelevee a la **fin** du train
(`validation_split` garde l'ordre, donc pas de fuite). On surveille l'ecart entre train et
validation : s'il se creuse, c'est du **sur-apprentissage**.

In [ ]:
history = model.fit(
    X_train, y_train,
    epochs=30,
    batch_size=32,
    validation_split=0.2,   # les 20 % les plus recents du train servent de validation
    verbose=1
)

In [ ]:
# Courbes d'apprentissage : train vs validation
plt.figure(figsize=(11, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='train')
plt.plot(history.history['val_loss'], label='validation')
plt.title('Perte (binary_crossentropy)')
plt.xlabel('epoque'); plt.legend(); plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='train')
plt.plot(history.history['val_accuracy'], label='validation')
plt.title('Accuracy')
plt.xlabel('epoque'); plt.legend(); plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

**Comment lire ces courbes ?** Si la perte de **train** baisse pendant que la perte de
**validation** remonte, le modele **sur-apprend** : il memorise le bruit du passe. Sur des donnees
de marche, c'est tres frequent — d'ou le Dropout et le faible nombre de parametres.

## 7. Evaluer : accuracy de test, matrice de confusion et BASELINE naive

On compare toujours le modele a une **baseline naive** : un modele « bete » qui predit toujours la
**classe majoritaire** (par ex. « toujours hausse »). Si notre LSTM ne fait pas mieux que ca, il
n'a **rien appris d'utile**.

In [ ]:
# Accuracy de test du LSTM
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print('Accuracy LSTM (test)        : %.4f' % test_acc)

# Predictions binaires (seuil 0.5)
proba = model.predict(X_test, verbose=0).reshape(-1)
y_pred = (proba > 0.5).astype('int32')

# --- BASELINE naive : toujours predire la classe majoritaire du TRAIN ---
majority_class = int(round(y_train.mean()))   # 1 si hausses majoritaires, sinon 0
baseline_pred = np.full_like(y_test, majority_class)
baseline_acc = accuracy_score(y_test, baseline_pred)

print('Classe majoritaire (train)  :', majority_class, '(1 = hausse, 0 = baisse)')
print('Accuracy BASELINE (test)    : %.4f' % baseline_acc)
print('Gain du LSTM vs baseline    : %+.4f' % (test_acc - baseline_acc))

In [ ]:
# Matrice de confusion du LSTM
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(5, 5))
ConfusionMatrixDisplay(cm, display_labels=['baisse (0)', 'hausse (1)']).plot(
    ax=ax, cmap='Blues', colorbar=False
)
plt.title('Matrice de confusion - LSTM (test)')
plt.tight_layout()
plt.show()

### Discussion honnete (esprit critique)

Prenez le temps de **lire et discuter** les points suivants — c'est le coeur pedagogique du TP.

**1. Pourquoi battre 50 % est-il si dur ?**
La direction quotidienne d'un marche est presque un **tirage a pile ou face**. Les acteurs
informes exploitent immediatement tout signal exploitable, ce qui le fait disparaitre (idee
proche de l'**hypothese des marches efficients**). Il reste donc tres peu de signal a capter :
gagner ne serait-ce que **1 a 5 points** au-dessus de la baseline est deja remarquable.

**2. Le LSTM bat-il vraiment la baseline ?**
Regardez le « Gain du LSTM vs baseline » ci-dessus. S'il est **proche de 0 voire negatif**, c'est
un resultat **normal et honnete** : sur des donnees reelles, un LSTM simple ne bat souvent **pas**
la baseline. Ce n'est pas un echec du TP — c'est une **lecon** sur la difficulte du probleme.

**3. Risque de sur-apprentissage.**
Avec assez d'epoques, le modele finit par memoriser le **bruit** du train (accuracy train qui
grimpe), sans que la validation/test ne s'ameliore. Le Dropout, la petite taille du modele et
l'arret precoce (early stopping) servent a limiter ce piege.

**4. L'illusion de la retro-prediction.**
Meme un bon score **sur le passe** ne prouve **rien** sur le futur. En pratique, des biais subtils
(data leakage, choix de la periode de test, frais de transaction ignores) gonflent les scores des
backtests. **Conclusion** : ce notebook enseigne le LSTM et l'esprit critique, **pas** une
strategie de trading. Ce n'est **pas un conseil d'investissement**.

## 8. A toi de jouer

Experimente et **note tes resultats** (config → accuracy test → gain vs baseline) :

1. **Change N** (la fenetre) : essaie `N = 5`, `N = 40`, `N = 60`. Une fenetre plus longue
   aide-t-elle, ou ajoute-t-elle surtout du bruit ?
2. **Teste un GRU** : remplace `layers.LSTM(64)` par `layers.GRU(64)`. Le GRU est un cousin du
   LSTM, plus leger. Compare accuracy et temps d'entrainement.
3. **Ajoute le volume** comme 2e feature : recupere `df_yf['Volume']`, calcule sa variation
   relative, et empile-la avec les rendements (passe a `features = 2` dans la forme d'entree).
4. **Change d'actif** : mets `TICKER = 'BTC-USD'` (crypto, plus volatile) ou `'AAPL'`, et
   re-execute. La crypto est-elle plus ou moins predictible ?
5. **Early stopping** : ajoute un `callbacks=[keras.callbacks.EarlyStopping(patience=5,
   restore_best_weights=True)]` dans `model.fit(...)` pour stopper avant le sur-apprentissage.

> Garde toujours en tete l'**AVERTISSEMENT** du debut : on apprend le Deep Learning,
> on ne fait pas de finance.

In [ ]:
# Ton espace d'experimentation
# Exemple : remplacer le LSTM par un GRU, changer N, ajouter un EarlyStopping...
# ...
